# Model 2 — TF-IDF + Logistic Regression (Classical ML)

**Proyek:** Deteksi Alergen pada Label Pangan Indonesia  
**Mata Kuliah:** COMP6885001 Natural Language Processing — BINUS University

---

## Deskripsi Model

Model 2 menggunakan pendekatan **classical machine learning** dengan:
- **TF-IDF** (Term Frequency-Inverse Document Frequency) untuk representasi teks
- **Logistic Regression** dengan strategi **OneVsRest** untuk klasifikasi multi-label

**Perbedaan dengan Model 1, 3, 4:**  
Model 2 adalah **document-level classification** — memprediksi *kategori alergen mana yang ada* dalam keseluruhan teks, bukan token spesifik mana yang merupakan alergen. Ini adalah pendekatan yang berbeda secara granularitas dari NER (Model 1, 3, 4).

**Pipeline:**
```
Teks Bahan → TF-IDF Vectorizer → [Skor TF-IDF] → OneVsRest LR → [Kategori Alergen]
```

**Tidak perlu GPU** — scikit-learn berjalan di CPU

In [ ]:
!pip install -q datasets pandas scikit-learn numpy

In [ ]:
import re
import json
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import pickle

random.seed(42)
np.random.seed(42)

## 1. Kamus Alergen & BIO Labeler

Sama dengan Model 1 — digunakan untuk menghasilkan label training (silver labels).

In [ ]:
ALLERGEN_CATEGORIES = {
    'GLUTEN': [
        'tepung terigu', 'tepung gandum', 'wheat starch', 'wheat bran', 'wheat germ',
        'malt extract', 'barley malt', 'hydrolyzed wheat protein',
        'tepung', 'terigu', 'gandum', 'gluten', 'barley', 'rye', 'oats',
        'sereal', 'roti', 'mie', 'pasta', 'biskuit', 'cracker', 'couscous',
        'bulgur', 'semolina', 'spelt', 'durum', 'kamut', 'malt', 'farina',
    ],
    'SUSU': [
        'susu sapi', 'milk powder', 'susu bubuk', 'nonfat milk', 'milk protein',
        'susu kental manis', 'whey protein', 'calcium caseinate', 'sodium caseinate',
        'rennet casein', 'milk solids', 'susu evaporasi', 'milk fat',
        'susu', 'keju', 'mentega', 'krim', 'yoghurt', 'butter', 'cream',
        'casein', 'kasein', 'whey', 'laktosa', 'lactose', 'skimmilk',
        'ghee', 'buttermilk', 'lactalbumin', 'lactoglobulin', 'dairy', 'caseinate',
    ],
    'TELUR': [
        'putih telur', 'kuning telur', 'egg white', 'egg yolk', 'egg powder',
        'telur bubuk', 'egg solids', 'egg protein', 'egg lecithin',
        'telur', 'egg', 'ovalbumin', 'ovomucoid', 'ovomucin',
        'lysozyme', 'mayonnaise', 'mayones', 'albumin', 'meringue',
    ],
    'KACANG': [
        'kacang tanah', 'peanut butter', 'kacang pohon', 'kacang mede', 'kacang mete',
        'kacang almond', 'kacang kenari', 'pine nut', 'brazil nut', 'tree nuts',
        'peanut', 'almond', 'cashew', 'walnut', 'pistachio', 'hazelnut',
        'pecan', 'macadamia', 'chestnut', 'groundnut', 'arachis',
    ],
    'KEDELAI': [
        'soy milk', 'susu kedelai', 'soy sauce', 'soy lecithin', 'soy protein',
        'soy isolate', 'soy flour', 'textured soy', 'lesitin kedelai', 'lesitin nabati',
        'soya lecithin',
        'kedelai', 'soy', 'soybean', 'tahu', 'tempe', 'tofu', 'tempeh',
        'edamame', 'miso', 'natto', 'yuba', 'okara',
        'lesitin', 'lecithin',
    ],
    'SEAFOOD': [
        'ikan teri', 'ikan tongkol', 'ikan kembung', 'ikan lele', 'ikan nila',
        'ikan bandeng', 'fish sauce', 'saus ikan', 'shrimp paste', 'fish oil',
        'krill oil', 'anchovy paste', 'saos tiram', 'saus tiram', 'oyster sauce',
        'ikan', 'tuna', 'salmon', 'sarden', 'makarel', 'cod', 'herring',
        'anchovy', 'udang', 'kerang', 'prawn', 'shrimp', 'cumi', 'sotong',
        'kepiting', 'crab', 'lobster', 'mussel', 'oyster', 'scallop',
        'abalone', 'terasi',
    ],
    'WIJEN': [
        'sesame oil', 'minyak wijen', 'biji wijen',
        'wijen', 'sesame', 'tahini',
    ],
    'SULFITE': [
        'sodium sulfite', 'potassium metabisulfite', 'sulfur dioxide',
        'natrium sulfit', 'kalium metabisulfit',
        'sulfite', 'sulfit',
    ],
}

ALL_CATEGORIES = list(ALLERGEN_CATEGORIES.keys())

def build_phrase_list(categories):
    phrases = []
    for cat, words in categories.items():
        for word in words:
            phrases.append((word.lower(), cat))
    phrases.sort(key=lambda x: len(x[0].split()), reverse=True)
    return phrases

PHRASE_LIST = build_phrase_list(ALLERGEN_CATEGORIES)

def tokenize_text(text):
    return re.findall(r'\w+|[^\w\s]', text.lower())

def bio_labeler(text):
    tokens = tokenize_text(text)
    labels = ['O'] * len(tokens)
    i = 0
    while i < len(tokens):
        matched = False
        for phrase, cat in PHRASE_LIST:
            phrase_tokens = tokenize_text(phrase)
            plen = len(phrase_tokens)
            if tokens[i:i + plen] == phrase_tokens:
                if all(labels[j] == 'O' for j in range(i, i + plen)):
                    labels[i] = f'B-{cat}'
                    for j in range(i + 1, i + plen):
                        labels[j] = f'I-{cat}'
                    i += plen
                    matched = True
                    break
        if not matched:
            i += 1
    return list(zip(tokens, labels))

def get_categories_from_bio(tokens_and_labels):
    return sorted({label[2:] for _, label in tokens_and_labels if label.startswith('B-')})

print(f'Kamus siap. Kategori: {ALL_CATEGORIES}')

## 2. Pengambilan & Pembuatan Data

Data training diambil dari dua sumber:
1. **Open Food Facts** (HuggingFace streaming, filter EN+ID)
2. **Data sintetis** — teks label makanan Indonesia yang dibuat dari kamus alergen

Silver label dihasilkan otomatis dari kamus menggunakan `bio_labeler()` → konversi ke **category-level** untuk training model klasifikasi.

In [ ]:
from datasets import load_dataset

# ==========================================
# KONFIGURASI
TOTAL_DATA  = 3000   # Jumlah sampel dari Open Food Facts
N_SYNTHETIC = 500    # Jumlah data sintetis
# ==========================================

print('Streaming Open Food Facts (filter EN+ID)...')
dataset = load_dataset('openfoodfacts/product-database', split='food', streaming=True)

sample_data = []
skipped = 0
for example in dataset:
    lang = example.get('lang', '')
    if lang not in ('en', 'id'):
        skipped += 1
        continue
    raw = example.get('ingredients_text')
    if not raw:
        continue
    if isinstance(raw, list) and len(raw) > 0:
        text = raw[0].get('text', '') if isinstance(raw[0], dict) else str(raw[0])
    else:
        text = str(raw)
    if not text or text == 'nan' or len(text) < 5:
        continue
    sample_data.append({'product_name': example.get('product_name', 'Unknown'), 'ingredients_text': text})
    if len(sample_data) >= TOTAL_DATA:
        break

df_off = pd.DataFrame(sample_data)
print(f'Terkumpul dari OFF  : {len(df_off)} produk')
print(f'Dilewati (bukan EN/ID): {skipped:,}')

In [ ]:
NON_ALLERGEN_WORDS = [
    'garam', 'air', 'gula', 'minyak sawit', 'pati', 'vitamin c',
    'kalium sorbat', 'asam sitrat', 'pewarna makanan', 'antioksidan',
    'pengatur keasaman', 'natrium klorida', 'glukosa', 'asam laktat',
    'karagenan', 'agar', 'pektin', 'karamel', 'natrium bikarbonat',
    'perisa', 'ekstrak vanili', 'kunyit',
]

TEMPLATES_ID = [
    'Komposisi: {bahan}.', 'Bahan: {bahan}.', 'Mengandung: {bahan}.',
    'Kandungan bahan: {bahan}.', '{bahan}.', 'Komposisi produk: {bahan}.',
]

def generate_synthetic_indonesian(n=500, seed=42):
    random.seed(seed)
    all_words = [(w, cat) for cat, words in ALLERGEN_CATEGORIES.items() for w in words]
    synthetic = []
    for _ in range(n):
        n_alr = random.randint(1, 3)
        n_non = random.randint(2, 5)
        selected_alr = random.sample(all_words, min(n_alr, len(all_words)))
        selected_non = random.sample(NON_ALLERGEN_WORDS, min(n_non, len(NON_ALLERGEN_WORDS)))
        ingredients = [w for w, _ in selected_alr] + selected_non
        random.shuffle(ingredients)
        text = random.choice(TEMPLATES_ID).format(bahan=', '.join(ingredients))
        synthetic.append({'product_name': 'Synthetic-ID', 'ingredients_text': text})
    return synthetic

df_synthetic = pd.DataFrame(generate_synthetic_indonesian(n=N_SYNTHETIC))
df_combined  = pd.concat([df_off, df_synthetic], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Total data gabungan: {len(df_combined)}')

## 3. Feature Engineering & Label Preparation

**TF-IDF (Term Frequency-Inverse Document Frequency):**
- Representasi statistik kata dalam teks
- Bobot tinggi → kata penting tapi tidak terlalu umum
- `ngram_range=(1,2)` → unigram + bigram untuk menangkap frase seperti "susu bubuk"
- `max_features=15000` → 15K fitur teratas berdasarkan frekuensi dokumen

**Label:** Konversi BIO → category-level binary vector  
`[has_GLUTEN, has_SUSU, has_TELUR, has_KACANG, has_KEDELAI, has_SEAFOOD, has_WIJEN, has_SULFITE]`

In [ ]:
# Generate silver labels (category-level) dari bio_labeler
print('Menghasilkan silver labels...')
texts, labels_list = [], []
for _, row in df_combined.iterrows():
    text = str(row['ingredients_text'])
    if not text or text == 'nan' or len(text) < 3:
        continue
    bio = bio_labeler(text)
    cats = get_categories_from_bio(bio)
    texts.append(text)
    labels_list.append(cats)

# MultiLabelBinarizer
mlb = MultiLabelBinarizer(classes=ALL_CATEGORIES)
Y   = mlb.fit_transform(labels_list)

label_counts = Y.sum(axis=0)
print(f'Total sampel: {len(texts)}')
print('Distribusi label:')
for cat, count in zip(ALL_CATEGORIES, label_counts):
    print(f'  {cat:<12}: {count:>6} sampel ({count/len(texts)*100:.1f}%)')

In [ ]:
# Train/test split
X_train, X_test, Y_train, Y_test = train_test_split(
    texts, Y, test_size=300, random_state=42
)

# TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    lowercase=True,
    min_df=2,
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Train  : {X_train_tfidf.shape[0]} sampel, {X_train_tfidf.shape[1]} fitur')
print(f'Test   : {X_test_tfidf.shape[0]} sampel')

# Top fitur untuk setiap kategori
feature_names = tfidf.get_feature_names_out()
print(f'\n5 unigram teratas dari kosakata TF-IDF (contoh):')
print(', '.join(feature_names[:5].tolist()))

## 4. Training — OneVsRest Logistic Regression

**OneVsRest** melatih satu classifier binary per kategori:
- Classifier GLUTEN: "mengandung GLUTEN atau tidak?"
- Classifier SUSU: "mengandung SUSU atau tidak?"
- ... (8 classifier total)

**Logistic Regression** dipilih karena:
- Interpretable (bobot fitur bisa dilihat)
- Efisien untuk sparse TF-IDF vectors
- Performa baik pada klasifikasi teks

In [ ]:
print('Training OneVsRest Logistic Regression...')

clf = OneVsRestClassifier(
    LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'),
    n_jobs=-1
)
clf.fit(X_train_tfidf, Y_train)

print('Training selesai!')

# Evaluasi pada test set silver
Y_pred_test = clf.predict(X_test_tfidf)
print()
print('=== Evaluasi pada Test Set Silver (300 sampel) ===')
print(classification_report(Y_test, Y_pred_test, target_names=ALL_CATEGORIES, zero_division=0))
silver_micro_f = f1_score(Y_test, Y_pred_test, average='micro', zero_division=0)
print(f'Silver Micro F1: {silver_micro_f:.4f}')

In [ ]:
# Tampilkan fitur paling informatif per kategori
print('=== Top 10 Fitur TF-IDF per Kategori ===')
for cat, estimator in zip(ALL_CATEGORIES, clf.estimators_):
    if hasattr(estimator, 'coef_'):
        top_idx = estimator.coef_[0].argsort()[-10:][::-1]
        top_features = [feature_names[i] for i in top_idx]
        print(f'  {cat:<12}: {top_features}')
    else:
        print(f'  {cat:<12}: (tidak tersedia)')

## 5. Demo Inferensi

In [ ]:
def predict_categories_m2(text):
    """Predict allergen categories using TF-IDF + Logistic Regression."""
    vec = tfidf.transform([text])
    pred = clf.predict(vec)[0]
    return sorted([ALL_CATEGORIES[i] for i, v in enumerate(pred) if v == 1])


DEMO_TEXTS = [
    'Komposisi: tepung terigu, gula, susu bubuk, kacang tanah, perisa alami.',
    'Bahan: tuna, shrimp paste, soy sauce, sesame oil, garam.',
    'Ingredients: peanut butter, milk powder, wheat starch, soy lecithin.',
    'Komposisi: gula, pati jagung, air, pewarna (E150a), tanpa alergen.',
    'Bahan: lesitin nabati, minyak sawit, coklat, susu kental manis, telur.',
]

print('=== Demo Inferensi Model 2 (TF-IDF + LR) ===')
for text in DEMO_TEXTS:
    cats = predict_categories_m2(text)
    print(f'Input    : {text[:70]}...' if len(text) > 70 else f'Input    : {text}')
    print(f'Prediksi : {cats if cats else "Tidak ada alergen"}')
    print()

## 6. Evaluasi — Gold Label (50 Produk Indonesia)

In [ ]:
EVAL_FILE = 'evaluation_dataset.json'

if not os.path.exists(EVAL_FILE):
    print(f'File {EVAL_FILE} tidak ditemukan.')
    print('Jalankan OCR + anotasi untuk membuat file evaluasi ini.')
    M2_RESULTS = None
else:
    with open(EVAL_FILE, 'r', encoding='utf-8') as f:
        eval_data = json.load(f)

    y_true_raw, y_pred_raw = [], []
    for item in eval_data:
        pred = predict_categories_m2(item['ingredient_text'])
        y_pred_raw.append(pred)
        y_true_raw.append(item['ground_truth'])

    mlb_eval = MultiLabelBinarizer(classes=ALL_CATEGORIES)
    Y_true   = mlb_eval.fit_transform(y_true_raw)
    Y_pred   = mlb_eval.transform(y_pred_raw)

    print('=== MODEL 2: TF-IDF + Logistic Regression (Gold Label Evaluation) ===')
    print(classification_report(Y_true, Y_pred, target_names=ALL_CATEGORIES, zero_division=0))

    micro_p = precision_score(Y_true, Y_pred, average='micro', zero_division=0)
    micro_r = recall_score(Y_true, Y_pred, average='micro', zero_division=0)
    micro_f = f1_score(Y_true, Y_pred, average='micro', zero_division=0)
    macro_f = f1_score(Y_true, Y_pred, average='macro', zero_division=0)

    print(f'Micro Precision : {micro_p:.4f}')
    print(f'Micro Recall    : {micro_r:.4f}')
    print(f'Micro F1        : {micro_f:.4f}')
    print(f'Macro F1        : {macro_f:.4f}')

    M2_RESULTS = {
        'model': 'TF-IDF + Logistic Regression',
        'micro_precision': round(micro_p, 4),
        'micro_recall': round(micro_r, 4),
        'micro_f1': round(micro_f, 4),
        'macro_f1': round(macro_f, 4),
    }
    print(f'\nM2_RESULTS = {M2_RESULTS}')

## 7. Simpan Model

In [ ]:
import pickle

with open('model2_tfidf.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('model2_clf.pkl', 'wb') as f:
    pickle.dump(clf, f)
with open('model2_mlb.pkl', 'wb') as f:
    pickle.dump(mlb, f)

print('Model 2 tersimpan: model2_tfidf.pkl, model2_clf.pkl, model2_mlb.pkl')

# Download (Colab)
try:
    from google.colab import files
    files.download('model2_tfidf.pkl')
    files.download('model2_clf.pkl')
    files.download('model2_mlb.pkl')
except ImportError:
    print('(Bukan Colab — file disimpan lokal)')

## 8. Analisis Hasil

### Kekuatan Model 2 (TF-IDF + LR)
- **Belajar dari data** — dapat menangkap pola statistik dari teks training
- **Lebih fleksibel** dari kamus — bisa memprediksi berdasarkan pola co-occurrence
- **Interpretable** — bobot fitur TF-IDF bisa dianalisis
- **Cepat** — training dan inference sama-sama cepat

### Kelemahan Model 2 (TF-IDF + LR)
- **Tidak menangkap semantik** — "susu" dan "milk" dianggap dua kata berbeda
- **Granularitas berbeda** — hanya category-level, tidak token-level NER
- **Terikat pada distribusi training** — tidak generalisasi ke kata yang tidak pernah dilihat
- **Silver label bias** — model belajar dari silver labels yang tidak sempurna

### Catatan Perbandingan
Model 2 mengerjakan **document-level classification** sedangkan Model 1, 3, 4 mengerjakan **token-level NER**. Keduanya dievaluasi pada **category-level F1** untuk perbandingan fair.

In [ ]:
print('=== RINGKASAN MODEL 2 ===')
print()
print('Model     : TF-IDF + OneVsRest Logistic Regression')
print('Task      : Multi-label text classification (document-level)')
print('Features  : TF-IDF unigram + bigram (max 15K fitur)')
print('Training  : Silver labels dari bio_labeler() pada OFF+Sintetis')
print(f'Train set : {len(X_train)} sampel')
print(f'Silver F1 : {silver_micro_f:.4f} (micro, test split silver)')
print()
if M2_RESULTS:
    print('--- Hasil Evaluasi Gold Label ---')
    for k, v in M2_RESULTS.items():
        if k != 'model':
            print(f'  {k:<22}: {v}')
else:
    print('[Hasil evaluasi gold label belum tersedia]')
print()
print('Keunggulan: Belajar dari data, lebih fleksibel dari kamus exact-match')
print('Kelemahan : Document-level saja, tidak menangkap semantik kata')